# Three-Body Problem: Unequal Mass at Scale
## Farey Nobility vs Binary Stability in Li-Liao 135K Catalog

**Goal:** For 1,349 orbits with free-group words, compute Gamma(2) matrices,
continued fraction nobility of spectral radius, and test whether nobility
predicts binary stability (S/U) better than word length.

**Runtime:** ~5-10 min (Quick), ~30 min (Full)

**GPU:** Not required (CPU-only). Uses Colab for fast internet download.

In [ ]:
# --- CELL 1: Install dependencies ---
!pip install -q mpmath scipy scikit-learn matplotlib

In [ ]:
# --- CELL 2: Configuration ---
# Set MODE = 'quick' for ~1 minute test, 'full' for complete analysis
MODE = 'quick'  # Change to 'full' for complete run

import os, json, time, re, warnings
import numpy as np
from fractions import Fraction
from math import gcd, isqrt, sqrt
from collections import Counter, defaultdict
import mpmath
mpmath.mp.dps = 100
warnings.filterwarnings('ignore')

QUICK_LIMIT = 100   # orbits in quick mode
FULL_LIMIT = 1349   # all orbits with free-group words
ORBIT_LIMIT = QUICK_LIMIT if MODE == 'quick' else FULL_LIMIT
print(f'Mode: {MODE}, processing up to {ORBIT_LIMIT} orbits')

In [ ]:
# --- CELL 3: Download Li-Liao catalog from GitHub ---
import urllib.request

BASE_URL = 'https://raw.githubusercontent.com/sjtu-liao/three-body/master/'
FILES = {
    'words': 'three-body-free-group-word.md',
    'initial_1': 'three-body-initial-condition-1.md',
    'initial_2': 'three-body-initial-condition-2.md',
    'periods': 'three-body-period.md',
}

os.makedirs('li_liao_data', exist_ok=True)

for name, fname in FILES.items():
    local = f'li_liao_data/{fname}'
    if not os.path.exists(local):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(BASE_URL + fname, local)
        print(f'  Saved to {local}')
    else:
        print(f'  {fname} already cached')

# Also try the supplementary 135K file if available
SUPP_URL = 'https://raw.githubusercontent.com/sjtu-liao/three-body/master/'
supp_files = [
    'three-body-unequal-mass-initial-condition.md',
    'three-body-unequal-mass-period.md',
]
for fname in supp_files:
    local = f'li_liao_data/{fname}'
    if not os.path.exists(local):
        try:
            urllib.request.urlretrieve(SUPP_URL + fname, local)
            print(f'  Downloaded supplementary: {fname}')
        except Exception as e:
            print(f'  Could not download {fname}: {e}')

print('\nDownload complete.')

In [ ]:
# --- CELL 4: Parse catalog ---

def parse_words_file(path):
    """Parse free-group words from the catalog markdown."""
    orbits = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or line.startswith('|'):
                # Try parsing markdown table rows
                if '|' in line and not line.startswith('#'):
                    parts = [p.strip() for p in line.split('|') if p.strip()]
                    if len(parts) >= 2 and parts[-1] and not parts[-1].startswith('-'):
                        word = parts[-1].strip()
                        if re.match(r'^[aAbB]+$', word):
                            orbits.append({'word': word, 'raw': parts})
                continue
            # Try direct parsing
            parts = line.split()
            if len(parts) >= 1:
                for p in parts:
                    if re.match(r'^[aAbB]+$', p) and len(p) > 1:
                        orbits.append({'word': p, 'raw': parts})
                        break
    return orbits

words_path = 'li_liao_data/three-body-free-group-word.md'
orbits_raw = parse_words_file(words_path)
print(f'Parsed {len(orbits_raw)} orbits with free-group words')
if orbits_raw:
    print(f'First 5 words: {[o["word"][:20] for o in orbits_raw[:5]]}')

In [ ]:
# --- CELL 5: Gamma(2) matrix computation and nobility ---

# Generator matrices for Gamma(2) < SL(2,Z)
GENERATORS = {
    'a': np.array([[1, 2], [0, 1]], dtype=np.int64),
    'A': np.array([[1, -2], [0, 1]], dtype=np.int64),
    'b': np.array([[1, 0], [2, 1]], dtype=np.int64),
    'B': np.array([[1, 0], [-2, 1]], dtype=np.int64),
}

def word_to_matrix(word):
    """Multiply generator matrices for the given word."""
    M = np.eye(2, dtype=np.int64)
    for ch in word:
        M = M @ GENERATORS[ch]
    return M

def spectral_radius(M):
    """Spectral radius of 2x2 integer matrix."""
    tr = abs(int(M[0,0] + M[1,1]))
    disc = tr*tr - 4*int(np.linalg.det(M))
    if disc < 0:
        return sqrt(abs(int(np.linalg.det(M))))
    return (tr + sqrt(disc)) / 2

def cf_expansion(x, max_terms=50):
    """Continued fraction expansion of x using mpmath."""
    x = mpmath.mpf(x)
    cf = []
    for _ in range(max_terms):
        a = int(mpmath.floor(x))
        cf.append(a)
        frac = x - a
        if frac < mpmath.mpf('1e-50'):
            break
        x = 1 / frac
    return cf

def nobility(cf):
    """Compute nobility = fraction of 1s in CF expansion.
    Noble numbers have all-1 tails (golden ratio family).
    Higher nobility -> more irrational, harder to approximate."""
    if len(cf) <= 1:
        return 0.0
    tail = cf[1:]  # skip integer part
    if not tail:
        return 0.0
    return sum(1 for a in tail if a == 1) / len(tail)

def free_reduce(word):
    """Freely reduce a word in F_2."""
    stack = []
    for ch in word:
        if stack and ((stack[-1] == 'a' and ch == 'A') or
                      (stack[-1] == 'A' and ch == 'a') or
                      (stack[-1] == 'b' and ch == 'B') or
                      (stack[-1] == 'B' and ch == 'b')):
            stack.pop()
        else:
            stack.append(ch)
    return ''.join(stack)

print('Functions defined. Processing orbits...')

In [ ]:
# --- CELL 6: Process all orbits ---
t0 = time.time()

results = []
for i, orb in enumerate(orbits_raw[:ORBIT_LIMIT]):
    word = orb['word']
    reduced = free_reduce(word)

    # Gamma(2) matrix
    try:
        M = word_to_matrix(reduced)
        tr = int(M[0,0] + M[1,1])
        det_M = int(round(np.linalg.det(M)))
        sr = spectral_radius(M)

        # CF of spectral radius
        cf = cf_expansion(sr)
        nob = nobility(cf)

        # Classify: hyperbolic (|tr| > 2), parabolic (|tr| = 2), elliptic
        if abs(tr) > 2:
            orbit_type = 'hyperbolic'
        elif abs(tr) == 2:
            orbit_type = 'parabolic'
        else:
            orbit_type = 'elliptic'

        results.append({
            'idx': i,
            'word': word[:30],
            'reduced_len': len(reduced),
            'word_len': len(word),
            'trace': tr,
            'det': det_M,
            'spectral_radius': float(sr),
            'cf_length': len(cf),
            'nobility': nob,
            'orbit_type': orbit_type,
            'matrix': M.tolist(),
        })
    except Exception as e:
        results.append({'idx': i, 'word': word[:30], 'error': str(e)})

    if (i+1) % 100 == 0:
        print(f'  Processed {i+1}/{min(len(orbits_raw), ORBIT_LIMIT)} orbits...')

elapsed = time.time() - t0
valid = [r for r in results if 'nobility' in r]
errors = [r for r in results if 'error' in r]
print(f'\nProcessed {len(results)} orbits in {elapsed:.1f}s')
print(f'  Valid: {len(valid)}, Errors: {len(errors)}')

In [ ]:
# --- CELL 7: Stability classification ---
# Binary stability proxy: hyperbolic orbits with |trace| > threshold are unstable
# This is a topological proxy -- true stability requires Lyapunov exponent data

# Classify stability based on orbit type
for r in valid:
    # Heuristic: hyperbolic with large |trace| -> unstable
    # Elliptic/parabolic -> stable
    if r['orbit_type'] == 'hyperbolic':
        r['stability'] = 'U'  # unstable
    else:
        r['stability'] = 'S'  # stable

n_stable = sum(1 for r in valid if r['stability'] == 'S')
n_unstable = sum(1 for r in valid if r['stability'] == 'U')
print(f'Stability classification (topological proxy):')
print(f'  Stable (S): {n_stable}')
print(f'  Unstable (U): {n_unstable}')

In [ ]:
# --- CELL 8: Nobility vs Stability Analysis ---
from scipy import stats

stable_nob = [r['nobility'] for r in valid if r['stability'] == 'S']
unstable_nob = [r['nobility'] for r in valid if r['stability'] == 'U']

print('=== NOBILITY vs STABILITY ===')
if stable_nob and unstable_nob:
    print(f'  Stable   nobility: mean={np.mean(stable_nob):.3f}, std={np.std(stable_nob):.3f}, n={len(stable_nob)}')
    print(f'  Unstable nobility: mean={np.mean(unstable_nob):.3f}, std={np.std(unstable_nob):.3f}, n={len(unstable_nob)}')
    t_stat, p_val = stats.ttest_ind(stable_nob, unstable_nob)
    print(f'  t-test: t={t_stat:.3f}, p={p_val:.4e}')
    u_stat, u_pval = stats.mannwhitneyu(stable_nob, unstable_nob, alternative='two-sided')
    print(f'  Mann-Whitney U: U={u_stat:.0f}, p={u_pval:.4e}')
else:
    print('  Insufficient data for comparison.')

# Word length vs stability
stable_wl = [r['word_len'] for r in valid if r['stability'] == 'S']
unstable_wl = [r['word_len'] for r in valid if r['stability'] == 'U']

print('\n=== WORD LENGTH vs STABILITY ===')
if stable_wl and unstable_wl:
    print(f'  Stable   word_len: mean={np.mean(stable_wl):.1f}, std={np.std(stable_wl):.1f}')
    print(f'  Unstable word_len: mean={np.mean(unstable_wl):.1f}, std={np.std(unstable_wl):.1f}')
    t2, p2 = stats.ttest_ind(stable_wl, unstable_wl)
    print(f'  t-test: t={t2:.3f}, p={p2:.4e}')

In [ ]:
# --- CELL 9: Logistic regression comparison ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

if len(valid) < 20:
    print('Too few orbits for logistic regression. Run in full mode.')
else:
    # Features
    X_nob = np.array([[r['nobility']] for r in valid])
    X_wl = np.array([[r['word_len']] for r in valid])
    X_both = np.array([[r['nobility'], r['word_len'], r['spectral_radius']] for r in valid])
    y = np.array([1 if r['stability'] == 'U' else 0 for r in valid])

    scaler = StandardScaler()

    print('=== LOGISTIC REGRESSION: Predicting Instability ===')
    for name, X in [('Nobility only', X_nob), ('Word length only', X_wl), ('All features', X_both)]:
        X_s = scaler.fit_transform(X)
        clf = LogisticRegression(max_iter=1000)
        scores = cross_val_score(clf, X_s, y, cv=min(5, len(valid)//5), scoring='accuracy')
        try:
            auc_scores = cross_val_score(clf, X_s, y, cv=min(5, len(valid)//5), scoring='roc_auc')
            print(f'  {name:20s}: Acc={scores.mean():.3f}+/-{scores.std():.3f}, AUC={auc_scores.mean():.3f}+/-{auc_scores.std():.3f}')
        except:
            print(f'  {name:20s}: Acc={scores.mean():.3f}+/-{scores.std():.3f}')

In [ ]:
# --- CELL 10: Visualization ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Nobility distribution by stability
ax = axes[0, 0]
if stable_nob:
    ax.hist(stable_nob, bins=20, alpha=0.6, label=f'Stable (n={len(stable_nob)})', color='blue')
if unstable_nob:
    ax.hist(unstable_nob, bins=20, alpha=0.6, label=f'Unstable (n={len(unstable_nob)})', color='red')
ax.set_xlabel('Nobility')
ax.set_ylabel('Count')
ax.set_title('Nobility Distribution by Stability')
ax.legend()

# 2. Nobility vs spectral radius
ax = axes[0, 1]
nobs = [r['nobility'] for r in valid]
srs = [np.log10(max(r['spectral_radius'], 1e-10)) for r in valid]
colors = ['red' if r['stability'] == 'U' else 'blue' for r in valid]
ax.scatter(nobs, srs, c=colors, alpha=0.4, s=10)
ax.set_xlabel('Nobility')
ax.set_ylabel('log10(Spectral Radius)')
ax.set_title('Nobility vs Spectral Radius')

# 3. Word length vs nobility
ax = axes[1, 0]
wls = [r['word_len'] for r in valid]
ax.scatter(wls, nobs, c=colors, alpha=0.4, s=10)
ax.set_xlabel('Word Length')
ax.set_ylabel('Nobility')
ax.set_title('Word Length vs Nobility')

# 4. Trace distribution
ax = axes[1, 1]
traces = [r['trace'] for r in valid]
ax.hist(traces, bins=50, alpha=0.7)
ax.set_xlabel('Trace')
ax.set_ylabel('Count')
ax.set_title('Trace Distribution')
ax.axvline(x=2, color='r', linestyle='--', label='|tr|=2')
ax.axvline(x=-2, color='r', linestyle='--')
ax.legend()

plt.tight_layout()
plt.savefig('3bp_nobility_analysis.png', dpi=150)
plt.show()
print('Plot saved to 3bp_nobility_analysis.png')

In [ ]:
# --- CELL 11: Summary and save results ---

summary = {
    'mode': MODE,
    'total_orbits': len(results),
    'valid_orbits': len(valid),
    'orbit_types': dict(Counter(r['orbit_type'] for r in valid)),
    'nobility_stats': {
        'mean': float(np.mean([r['nobility'] for r in valid])),
        'std': float(np.std([r['nobility'] for r in valid])),
        'min': float(np.min([r['nobility'] for r in valid])),
        'max': float(np.max([r['nobility'] for r in valid])),
    },
    'stability_split': {'S': n_stable, 'U': n_unstable},
}

print('\n' + '='*60)
print('SUMMARY')
print('='*60)
for k, v in summary.items():
    print(f'  {k}: {v}')

# Save
with open('3bp_unequal_mass_results.json', 'w') as f:
    json.dump({'summary': summary, 'orbits': results}, f, indent=2, default=str)
print('\nResults saved to 3bp_unequal_mass_results.json')

# Try saving to Google Drive if mounted
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import shutil
    dest = '/content/drive/MyDrive/Farey_Results/'
    os.makedirs(dest, exist_ok=True)
    shutil.copy('3bp_unequal_mass_results.json', dest)
    shutil.copy('3bp_nobility_analysis.png', dest)
    print(f'Results copied to Google Drive: {dest}')
except:
    print('Google Drive not mounted. Results saved locally only.')
    print('To save: mount Drive and copy files manually.')